# Sector Expenditure Profiling
Owner: Razz
Table: `pondolens.dev_bronze.sector_expenditure`

Baseline: 162 rows (81 per source file, 2024 and 2025)

## Goal
Identify comparable sector records for Q1. Do not modify Bronze.

## Checks
- Compare source file counts with Bronze.
- Check column types, nulls/blanks, and sample values against the source.
- Verify uniqueness by year, price basis, accounting basis, sector, and particulars.
- Verify years, sector labels, particulars, numeric amounts, and units.
- Keep different accounting/price bases separate.
- Identify sectors, subsectors, and totals to avoid double counting.
- Verify the denominator and scale of `percent_distribution`.

## How to read the result
The cell below shows 2 tables.
- **Source Check**: one row per check. `Rows Checked` is how many rows were looked at, `Bronze` is the result in Bronze, `Source` is the same result in the original Volume files (2024 and 2025 combined). Counts are rows affected, so 0 is good (except `Table Row Count`). The last rows compare Bronze with the source files: the `Source` cell is empty and `Bronze` is how many rows differ.
- **Data Types**: each Bronze column and its current data type.

In [0]:
import numpy as np
import pandas as pd

TABLE = "sector_expenditure"
BRONZE = "pondolens.dev_bronze.sector_expenditure"

SOURCE_PATH = "/Volumes/pondolens/dev_bronze/government_budget/source/02_sector_expenditure/"
SOURCE = [
    SOURCE_PATH + "02_sector_expenditure_2024.csv",
    SOURCE_PATH + "02_sector_expenditure_2025.csv",
]

KEY = ["year", "price_basis", "accounting_basis", "sector", "particulars"]
NUMERIC = ["year", "source_page", "expenditure_amount", "percent_distribution"]
DASHES = ["-", "--", "–", "—"]

sdf = spark.table(BRONZE)
bronze = sdf.toPandas()
types = dict(sdf.dtypes)

source = pd.concat(
    [pd.read_csv(p, keep_default_na=False, na_values=[""]) for p in SOURCE],
    ignore_index=True,
)
data_cols = list(source.columns)

for col in NUMERIC:
    bronze[col] = pd.to_numeric(bronze[col], errors="coerce")
    source[col] = pd.to_numeric(source[col], errors="coerce")


# each count is the number of rows affected, not cells
def profile(df):
    df = df[data_cols]
    blank = lambda s: s.isna() | (s.astype(str).str.strip() == "")
    zero = df.apply(lambda c: pd.to_numeric(c, errors="coerce") == 0)
    dash = df.apply(lambda c: c.notna() & c.astype(str).str.strip().isin(DASHES))
    return {
        "Table Row Count": len(df),
        "Sector Empty Count": blank(df["sector"]).sum(),
        "Particulars Empty Count": blank(df["particulars"]).sum(),
        "Null Count": df.isna().any(axis=1).sum(),
        "Zero Count": zero.any(axis=1).sum(),
        "Dash Count": dash.any(axis=1).sum(),
        "Exact Duplicate Rows": df.duplicated(keep=False).sum(),
        "Composite Key Duplicates (Year, Price Basis, Accounting Basis, Sector, Particulars)": df.duplicated(KEY, keep=False).sum(),
    }


def same(a, b):
    return (a == b) | (a.isna() & b.isna())


rows = []
b = profile(bronze)
s = profile(source)
for check, value in b.items():
    rows.append([TABLE, check, len(bronze), value, s.get(check)])

# bronze vs source: the same row is matched on the composite key
m = source.merge(bronze[data_cols], on=KEY, how="outer", indicator=True, suffixes=("_src", "_brz"))
both = m[m["_merge"] == "both"]
rows.append([TABLE, "Rows In Source Not In Bronze", len(source), (m["_merge"] == "left_only").sum(), None])
rows.append([TABLE, "Rows In Bronze Not In Source", len(bronze), (m["_merge"] == "right_only").sum(), None])

for col, label in [
    ("source_name", "Source Name"),
    ("source_page", "Source Page"),
    ("expenditure_amount", "Expenditure Amount"),
    ("unit_amount", "Unit Amount"),
    ("percent_distribution", "Percent Distribution"),
]:
    x, y = both[col + "_src"], both[col + "_brz"]
    if col in NUMERIC:
        diff = ~np.isclose(x.astype(float), y.astype(float), rtol=1e-9, equal_nan=True)
    else:
        diff = ~same(x.astype("object"), y.astype("object"))
    rows.append([TABLE, f"{label} Mismatch", len(both), diff.sum(), None])

checks = pd.DataFrame(rows, columns=["Table", "Check", "Rows Checked", "Bronze", "Source"])
for col in ["Rows Checked", "Bronze", "Source"]:
    checks[col] = checks[col].astype("Int64")

column_types = pd.DataFrame(
    [[TABLE, col, dtype] for col, dtype in types.items()],
    columns=["Table", "Column", "Bronze Data Type"],
)

print("Source Check")
display(checks)

print("Data Types")
display(column_types)

## Findings
Result of the checks on the Bronze table, compared with the original CSV files in the Volume. Bronze and the source have 162 rows.

| Table | Issue | Affected rows | Rows checked | Proposed Silver treatment |
|---|---|---|---|---|
| sector_expenditure | `particulars` is empty (NET LENDING, DEBT-SERVICE-INTEREST PAYMENTS and GRAND TOTAL, 9 rows per file) | 18 | 162 | These sectors have no subsector. Fill with a label (for example `All`, as the other sector totals use) or keep null. Needs to be reviewed. |

- Table Row Count is 162 in Bronze and in the source volume.
- Sector Empty, Zero, Dash, Exact Duplicate Rows and Composite Key Duplicates (Year, Price Basis, Accounting Basis, Sector, Particulars) are all 0.
- The 18 rows with a null are the same 18 rows with an empty `particulars`. The source files have the same 18.
- Bronze rows match the source file rows.
- 0 differences in `source_name`, `source_page`, `expenditure_amount`, `unit_amount` and `percent_distribution`.

## Data Types (Bronze)
| Table | Observation | Proposed Silver treatment |
|---|---|---|
| sector_expenditure | All source columns are `string`, including `year`, `source_page`, `expenditure_amount` and `percent_distribution`. | Cast `year` and `source_page` to INT, `expenditure_amount` to BIGINT and `percent_distribution` to DECIMAL(5,2) or FLOAT. NEeds to be reviewed. |
| sector_expenditure | Bronze has 3 extra columns that are not in the CSV files: `_rescued_data` (string), `_source_file` (string) and `_ingested_at` (timestamp). | Keep as load metadata. The checks above use the CSV columns only. |
| sector_expenditure | `source_name` and `_source_file` both hold a file name. | Only keep one. |

## Assumptions to Verify
| Interpretation / Question | Evidence | Status |
|---|---|---|
| Bronze has the same rows and values as the CSV files in the Volume | 162 rows on both sides, 0 rows on only one side, 0 differences in all compared columns | Confirmed |
| `percent_distribution` is the share of the GRAND TOTAL of the same file and price basis | Amount ÷ GRAND TOTAL × 100 matches the column for every row (largest gap 0.005, rounding) | Confirmed |
| Subsector rows add up to the sector `All` row | Within 1 million pesos (rounding) for ECONOMIC, SOCIAL and GENERAL PUBLIC SERVICES | Supported |
| `actual`, `program` and `proposed` in `price_basis` are budget stages, not price levels | One `accounting_basis` (`cash-based`) and one `unit_amount` (`million pesos`) in the data. The column name is confusing. | Pending |
| The same year in the 2024 and 2025 files is comparable | 2025 and 2026 appear in both files with a different `price_basis` (program/proposed vs actual/program) | Pending |

## Semantic Notes

- Keep `actual`, `program` and `proposed` separate. Choose one per analysis. `Actual` is the only needed one for silver.
- `All` rows are the sector total and the subsector rows are its breakdown. Do not add them together.
- DEFENSE `All` and `Domestic Security` have the same amount. Count only one.
- GRAND TOTAL, NET LENDING and DEBT-SERVICE-INTEREST PAYMENTS have no `particulars`. Do not add GRAND TOTAL to the sectors
- Sector share = comparable sector amount ÷ comparable total × 100.
- Document the spending measure, units, denominator, and aggregation level.
- Keep different accounting/price bases separate.
-  `expenditure_amount` = million pesos

## Conclusion
- Required Silver changes:
  1. Decide how to fill `particulars` for NET LENDING, DEBT-SERVICE-INTEREST PAYMENTS and GRAND TOTAL.
  2. Cast `year`, `source_page`, `expenditure_amount` and `percent_distribution` to numeric types.
  4. Only get the rows with actual as `price_basis`since that is the only needed data and program and proposed are still initial budget proposals and not the actual spending.
  5. Check the unit(million pesos)) to properly convert current amount.

- Ready for Q1? Yes but needs to be reviewed and rechecked according to findings.
- Open questions:
  1. What should `particulars` be for the sectors without a subsector?